# Baqaee & Farhi (2022) — COVID-19 Application: Data Layer

This notebook documents the data-preparation pipeline for replicating
Baqaee & Farhi (2022), *Supply and Demand in Disaggregated Keynesian
Economies with an Application to the COVID-19 Crisis* (AEA P&P).

**Source:** `RepAEA2022/Replication code_ver2/`

**Pipeline (3 stages):**
1. **IO Table** — load `IO_data_2018.mat` (BEA 66-sector Use table, year 2015),
   clean, compute `Ω` (row-normalized), `αL, αK` (factor shares),
   `β` (consumption weights), `va_share` / `int_share`.
2. **Shocks** — load the four `.xlsx` files with COVID-19 sectoral shocks
   (BLS labor, PCE demand, wage changes, PPI).
3. **Standard-Form Network** — relabel the IO table into a `D = 5N+4 = 334`
   input-output matrix `Ω_re` with factor codes, rigidity indicators, and
   Leontief inverse `Ψ = (I − Ω_re)⁻¹`.

In [13]:
# Setup: resolve project root from the active Project.toml
# (robust across VSCode kernels where @__DIR__ points to the wrong folder)
cd(dirname(Base.active_project()))
using Printf, LinearAlgebra, Statistics
include("src/io_table.jl")
include("src/shocks.jl")
include("src/network.jl")
include("src/model.jl")

DATA_DIR = "data"
N = 66      # 66 BEA sectors
YEAR = 2015 # base year (per Master_file_3.m)

println("Modules loaded — project root: $(pwd())");


[ Info: Precompiling NLsolve [2774e3e8-f4cf-5e23-947b-6d7e65073b56](cache misses: incompatible header (1))
[ Info: Precompiling NLsolve [2774e3e8-f4cf-5e23-947b-6d7e65073b56] (cache misses: incompatible header (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


Modules loaded — project root: /Users/joko/Git/BFRep/(3)BeyondHulten/bf_replication2


---
## 1. IO Table

`IO_data_2018.mat` contains the BEA 2018 Input-Output table as a 3-D array
`Data_raw[row, col, year_index]` with 22 years (1997–2018) and 83 sectors.
We use `year=2015` (index 19), keep the first 66 sectors, and follow the
column layout from `Master_file_3.m`.

The calibration:
- Remove first 2 columns (label metadata)
- Extract labor (row 77) and gross operating surplus / capital (row 79)
- `grossout = intermediate + labor + GOS`
- `Ω = row-normalize` the N×N intermediate block
- `αL = L / (L+K)`, `αK = K / (L+K)`
- `β = Final / sum(Final)` from Data_raw column 99 (year < 2016)

In [14]:
@printf("Loading IO table for N=%d, year=%d\n", N, YEAR)
io = load_io_table(joinpath(DATA_DIR, "IO_data_2018.mat"); N=N, year=YEAR);

Loading IO table for N=66, year=2015
Loading IO table from data/IO_data_2018.mat ... done. N=66, year=2015, beta sum=1.000000


In [15]:
# Overview of calibrated parameters
@printf("Omega: %d × %d  (row-normalized IO coefficients)\n", size(io.Omega, 1), size(io.Omega, 2))
@printf("beta:  %d sectors, sum = %.10f\n", length(io.beta), sum(io.beta))
@printf("alphaL: %d sectors, min=%.4f, max=%.4f\n", length(io.alphaL),
    minimum(io.alphaL), maximum(io.alphaL))
@printf("alphaK: %d sectors, min=%.4f, max=%.4f\n", length(io.alphaK),
    minimum(io.alphaK), maximum(io.alphaK))
@printf("va_share:  min=%.4f, max=%.4f\n", minimum(io.va_share), maximum(io.va_share))
@printf("int_share: min=%.4f, max=%.4f\n", minimum(io.int_share), maximum(io.int_share))

Omega: 66 × 66  (row-normalized IO coefficients)
beta:  66 sectors, sum = 1.0000000000
alphaL: 66 sectors, min=0.0114, max=0.9888
alphaK: 66 sectors, min=0.0112, max=0.9886
va_share:  min=0.2379, max=0.8969
int_share: min=0.1031, max=0.7621


In [16]:
# Sector names for the first 66 (indname has 83 total)
println("First 15 sector names:")
for i in 1:15
    @printf("  %3d  %s\n", i, io.indname[i])
end
println("  ...")

# Beta: which sectors have positive consumption shares
pos_idx = findall(x -> x > 0, io.beta)
println("\nSectors with positive beta (consumption share): $(length(pos_idx)) / $(N)")
println("Top-10 by consumption share:")
top10 = sortperm(io.beta, rev=true)[1:10]
for i in top10
    @printf("  %3d  %-55s  β = %.4f\n", i, io.indname[i], io.beta[i])
end

First 15 sector names:
    1  Farms
    2  Forestry, fishing, and related activities
    3  Oil and gas extraction
    4  Mining, except oil and gas
    5  Support activities for mining
    6  Utilities
    7  Construction
    8  Wood products
    9  Nonmetallic mineral products
   10  Primary metals
   11  Fabricated metal products
   12  Machinery
   13  Computer and electronic products
   14  Electrical equipment, appliances, and components
   15  Motor vehicles, bodies and trailers, and parts
  ...

Sectors with positive beta (consumption share): 61 / 66
Top-10 by consumption share:
   48  Housing Services                                         β = 0.1224
    7  Construction                                             β = 0.0758
   59  Hospitals                                                β = 0.0617
   58  Ambulatory health care services                          β = 0.0606
   27  Wholesale trade                                          β = 0.0537
   31  Other retail            

In [17]:
# Check Omega structure: row sums (should be 1 for all)
row_sums = sum(io.Omega, dims=2)
@printf("Omega row sums — min: %.6f, max: %.6f\n", minimum(row_sums), maximum(row_sums))

# VA and intermediate share summary
println("\nVA share vs Intermediate share by sector:")
for i in [1, 15, 30, 45, 66]
    @printf("  %3d  %-55s  va=%.3f  int=%.3f\n", i, io.indname[i], io.va_share[i], io.int_share[i])
end

println("\n✅ IO table calibration complete.")

Omega row sums — min: 1.000000, max: 1.000000

VA share vs Intermediate share by sector:
    1  Farms                                                    va=0.347  int=0.653
   15  Motor vehicles, bodies and trailers, and parts           va=0.238  int=0.762
   30  General merchandise stores                               va=0.594  int=0.406
   45  Securities, commodity contracts, and investments         va=0.513  int=0.487
   66  Other services, except government                        va=0.608  int=0.392

✅ IO table calibration complete.


---
## 2. COVID-19 Shocks

Four shock/outcome vectors are read from pre-built `.xlsx` files
(sourced from BLS, BEA, and Census data by `Master_file_1.R` and
`Master_file_2.do`):

| File | Column | What | Symbol |
|------|--------|------|--------|
| `BLS_labor_shock_202108.xlsx` | 4 = `diff_2005` | Supply: hours change (Feb–May 2020) | `A` |
| `PCE_shock_202107.xlsx` | 4 = `diff_2005_pce` | Demand: PCE spending change | `B` |
| `wage_change_final.xlsx` | 6 = `w_adj_20Q1_20Q2` | Wage change (Q1→Q2 2020) | validation |
| `ppi_data.xlsx` | 5 = `av_p_change_Feb_May` | PPI change (Feb→May 2020) | validation |

The HS (Housing) sector receives ORE (Other Real Estate)'s PPI value
since both map to NAICS 531.

In [18]:
shocks = load_shocks(DATA_DIR; N=N);

# Helper to print sector-level summary
function print_shock_table(name, vec, io, k=10)
    @printf("\n%s (min=%.4f, max=%.4f, mean=%.4f):\n", name,
        minimum(vec), maximum(vec), mean(vec))
    idx = sortperm(vec, by=x->abs(x), rev=true)[1:k]
    for i in idx
        @printf("  %3d  %-55s  %+.4f\n", i, io.indname[i], vec[i])
    end
end

print_shock_table("BLS labor shock (supply)", shocks.BLS_shock, io, 12)
print_shock_table("PCE demand shock", shocks.PCE_shock, io, 10)
print_shock_table("Wage change", shocks.wages, io, 10)
print_shock_table("PPI change", shocks.PPI, io, 10)

Loading shocks from data ...
  BLS shock: min=-0.5409, max=0.0206
  PCE shock: min=-0.9182, max=0.2089

BLS labor shock (supply) (min=-0.5409, max=0.0206, mean=-0.1310):
   63  Amusements, gambling, and recreation industries          -0.5409
   41  Motion picture and sound recording industries            -0.5383
   64  Accommodation                                            -0.4869
   62  Performing arts, spectator sports, museums, and related activities  -0.4444
   36  Transit and ground passenger transportation              -0.3937
   65  Food services and drinking places                        -0.3727
   21  Apparel and leather and allied products                  -0.2850
   32  Air transportation                                       -0.2571
   15  Motor vehicles, bodies and trailers, and parts           -0.2513
   16  Other transportation equipment                           -0.2513
    5  Support activities for mining                            -0.2504
   31  Other retail        

In [19]:
# Sector mapping: which industries were hit hardest?
# Sort BLS shock (supply) — most negative = biggest hours drop
println("\nTop-5 hardest-hit by BLS supply shock:")
for i in sortperm(shocks.BLS_shock)[1:5]
    @printf("  %3d  %-55s  BLS=%+.4f  PCE=%+.4f\n",
        i, io.indname[i], shocks.BLS_shock[i], shocks.PCE_shock[i])
end

println("\nTop-5 hardest-hit by PCE demand shock:")
for i in sortperm(shocks.PCE_shock)[1:5]
    @printf("  %3d  %-55s  BLS=%+.4f  PCE=%+.4f\n",
        i, io.indname[i], shocks.BLS_shock[i], shocks.PCE_shock[i])
end

println("\n✅ Shock data loaded.")


Top-5 hardest-hit by BLS supply shock:
   63  Amusements, gambling, and recreation industries          BLS=-0.5409  PCE=-0.7380
   41  Motion picture and sound recording industries            BLS=-0.5383  PCE=-0.3219
   64  Accommodation                                            BLS=-0.4869  PCE=-0.7716
   62  Performing arts, spectator sports, museums, and related activities  BLS=-0.4444  PCE=-0.7581
   36  Transit and ground passenger transportation              BLS=-0.3937  PCE=-0.7486

Top-5 hardest-hit by PCE demand shock:
   34  Water transportation                                     BLS=-0.1385  PCE=-0.9182
   32  Air transportation                                       BLS=-0.2571  PCE=-0.8940
   64  Accommodation                                            BLS=-0.4869  PCE=-0.7716
   62  Performing arts, spectator sports, museums, and related activities  BLS=-0.4444  PCE=-0.7581
   36  Transit and ground passenger transportation              BLS=-0.3937  PCE=-0.7486

✅ Shock

---
## 3. Standard-Form Network

The IO table is "relabeled" into a standard form (Section III of B&F 2020)
that separates goods, value-added, intermediates, labor, capital, and
consumers into distinct row/column blocks.

**Layout** (D = 5N + 4 = 334 for N=66):

```
Block         Rows          Contents               factor  keynes
─────         ────          ────────               ──────  ──────
Consumption      1          final demand           —       —
Goods          2..67        N goods               1       1
VA            68..133       N value-added         1       1
Intermediates 134..199      N intermediate         1       1
Labor         200..265      N labor factors        0      -1 (sticky)
Capital       266..331      N capital factors      0       0 (flexible)
HtM            332          HtM consumer           3       —
Ricardian      333          Ricardian consumer     2       —
Tomorrow       334          consumption good       0       0 (numeraire)
```

The Leontief inverse `Ψ = (I − Ω_re)⁻¹` gives the propagation of shocks
through the production network. The first row of Ψ contains **Domar
weights**: each sector's total (direct + indirect) share of GDP.

In [20]:
sf = build_standard_form(io)

@printf("D = %d (expected %d = 5×%d+4)\n", sf.D, 5*N+4, N)
@printf("Omega_re: %d × %d\n", size(sf.Omega_re, 1), size(sf.Omega_re, 2))
@printf("Psi_re:   %d × %d\n", size(sf.Psi_re, 1), size(sf.Psi_re, 2))

# Verify: density of Omega_re
nnz = count(x -> x > 0, sf.Omega_re)
@printf("Omega_re non-zeros: %d / %d (%.2f%% dense)\n",
    nnz, length(sf.Omega_re), 100 * nnz / length(sf.Omega_re))

Building standard form Ω_re: N=66, D=334
  Psi_re[1,2:N+1] sum = 1.807659
D = 334 (expected 334 = 5×66+4)
Omega_re: 334 × 334
Psi_re:   334 × 334
Omega_re non-zeros: 3819 / 111556 (3.42% dense)


In [21]:
# Factor and Keynes breakdown
println("\nFactor type counts:")
for (code, label) in [(1,"goods"), (0,"factors"), (2,"Ricardian"), (3,"HtM")]
    @printf("  %-12s  %d\n", label, count(x -> x == code, sf.factor))
end

println("\nKeynes rigidity counts:")
for (code, label) in [(1,"normal (CES good)"), (0,"flexible (capital/tomorrow)"), (-1,"sticky (labor)")]
    @printf("  %-30s  %d\n", label, count(x -> x == code, sf.keynes))
end


Factor type counts:
  goods         199
  factors       133
  Ricardian     1
  HtM           1

Keynes rigidity counts:
  normal (CES good)               201
  flexible (capital/tomorrow)     67
  sticky (labor)                  66


In [22]:
# Domar weights: first row of Psi_re for goods sectors (cols 2:N+1)
domar = sf.Domar
@printf("\nDomar weights (Psi_re[1, 2:%d]):\n", N+1)
@printf("  Sum = %.6f  (gross-output / GDP ratio, expected ~1.8–2.0)\n", sum(domar))
@printf("  Min = %.6f, Max = %.6f\n", minimum(domar), maximum(domar))

println("\nTop-10 sectors by Domar weight:")
idx = sortperm(domar, rev=true)[1:10]
for i in idx
    @printf("  %3d  %-55s  λ = %.4f  (β = %.4f)\n",
        i, io.indname[i], domar[i], io.beta[i])
end


Domar weights (Psi_re[1, 2:67]):
  Sum = 1.807659  (gross-output / GDP ratio, expected ~1.8–2.0)
  Min = 0.002146, Max = 0.122385

Top-10 sectors by Domar weight:
   48  Housing Services                                         λ = 0.1224  (β = 0.1224)
   53  Miscellaneous professional, scientific, and technical services  λ = 0.1115  (β = 0.0406)
   27  Wholesale trade                                          λ = 0.0950  (β = 0.0537)
    7  Construction                                             λ = 0.0887  (β = 0.0758)
   49  Other Real Estate                                        λ = 0.0709  (β = 0.0080)
   58  Ambulatory health care services                          λ = 0.0624  (β = 0.0606)
   59  Hospitals                                                λ = 0.0619  (β = 0.0617)
   46  Insurance carriers and related activities                λ = 0.0600  (β = 0.0207)
   19  Food and beverage and tobacco products                   λ = 0.0583  (β = 0.0351)
   31  Other retail         

In [23]:
# Psi_re invertibility check
resid = norm(sf.Psi_re * (I - sf.Omega_re) - I)
@printf("\nPsi_re invertibility check:\n")
@printf("  ||Psi_re * (I - Omega_re) - I|| = %.2e\n", resid)
if resid < 1e-10
    println("  ✅ Invertibility holds — network is consistent.")
else
    println("  ⚠️  Residual > 1e-10 — check construction.")
end


Psi_re invertibility check:
  ||Psi_re * (I - Omega_re) - I|| = 4.05e-15
  ✅ Invertibility holds — network is consistent.


---
## 4. Verification Summary

All checks pass. The data layer is ready for the equilibrium solver
(`src/model.jl`, NLsolve + Fischer–Burmeister — see `02_equilibrium.ipynb`).

| Check | Result |
|-------|--------|
| N = 66 sectors | ✅ |
| beta sums to 1 | ✅ |
| alphaL + alphaK = 1 for all sectors | ✅ |
| Omega row-normalized (sum ≈ 1 per row) | ✅ |
| BLS shock: −54% to +2.1% | ✅ |
| PCE shock: −91.8% to +20.9% | ✅ |
| D = 5N + 4 = 334 | ✅ |
| 61/66 sectors have positive beta | ✅ |
| Psi_re invertible (resid < 1e-12) | ✅ |
| factor/keynes vectors correct dimensions | ✅ |
| 66 sticky (labor) + 67 flex (capital + tomorrow) | ✅ |


In [24]:
println("✅ All checks passed — data layer is ready for the equilibrium solver (Phase 4).")


✅ All checks passed — data layer is ready for the equilibrium solver (Phase 4).
